[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ohtopico/site/blob/main/notebooks/wikipedia/02-previsao-de-series.ipynb)

# Previsão de séries temporais: treino, teste e explicabilidade

Dá para prever quantas pessoas vão ler um verbete no mês que vem? Neste notebook você separa os dados em treino e teste, compara três modelos, olha para dentro do modelo para entender cada parâmetro e testa se a diferença de erro entre dois modelos é real ou fruto do acaso.

**Público:** professores, ensino médio, cursos técnicos e graduação · **Tempo:** 2 aulas de 50 min · **BNCC:** EF09MA22, EF09CO10 (ensino médio: EM13MAT406)

Notebook de [Júlia Roberta Quoos Alves](https://ohtopico.com.br/curriculo/) · [ohtopico.com.br](https://ohtopico.com.br/interesse-por-ia/notebooks/) · Licença CC BY-NC-SA 4.0

**Como usar:** clique em cada célula de código e aperte o botão de play (ou Shift + Enter), de cima para baixo. Os campos ao lado das células (listas e caixas de texto) podem ser trocados à vontade: depois de mudar, rode a célula de novo.

## Objetivos
- Entender por que um modelo é avaliado em dados que ele não viu (treino e teste).
- Comparar modelos com uma medida de erro.
- Ler os parâmetros de um modelo simples e explicar o que cada um faz.
- Usar uma semente (seed) para tornar um resultado aleatório reproduzível.
- Testar se uma diferença de desempenho é estatisticamente significativa.

In [ ]:
# @title Configuração (rode primeiro) { display-mode: "form" }
# Funções para baixar os dados públicos da Wikimedia. Não precisa mudar nada aqui.
import json, re, time, urllib.parse
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

CORES = ['#157f3b', '#b3123a', '#c99a00', '#111111', '#5fb27c', '#7a1f33']
plt.rcParams.update({'figure.figsize': (11, 4.5), 'axes.prop_cycle': plt.cycler(color=CORES), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .3, 'font.size': 11})
pd.options.display.float_format = '{:,.0f}'.format
UA = {'User-Agent': 'notebooks-ohtopico/1.0 (https://ohtopico.com.br; contato@ohtopico.com.br)'}
FONTE = 'Fonte: Wikimedia, visitas de pessoas à Wikipédia em português'


def _get(url, params=None):
    r = requests.get(url, params=params, headers=UA, timeout=60)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()


def titulo_oficial(titulo):
    """Corrige maiúsculas e segue redirecionamentos: 'ia' vira 'Inteligência artificial'."""
    j = _get('https://pt.wikipedia.org/w/api.php', dict(action='query', titles=titulo, redirects=1, format='json', formatversion=2))
    pagina = j['query']['pages'][0]
    if pagina.get('missing'):
        raise ValueError(f'Não achei o verbete "{titulo}" na Wikipédia em português. Confira a grafia.')
    return pagina['title']


def visitas(titulos, inicio='2016-01-01', fim=None, por='mes'):
    """Tabela com as visitas de cada verbete. por='mes' ou por='dia'."""
    if isinstance(titulos, str):
        titulos = [titulos]
    fim = pd.Timestamp(fim) if fim else pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
    if por == 'mes':
        fim = fim.replace(day=1) - pd.Timedelta(days=1)  # só meses completos
    colunas = {}
    for t in titulos:
        t = titulo_oficial(t)
        art = urllib.parse.quote(t.replace(' ', '_'), safe='')
        g = 'monthly' if por == 'mes' else 'daily'
        url = (f'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/pt.wikipedia/all-access/user/{art}/{g}/'
               f'{pd.Timestamp(inicio):%Y%m%d}00/{fim:%Y%m%d}00')
        j = _get(url) or {'items': []}
        colunas[t] = pd.Series({pd.Timestamp(i['timestamp'][:8]): i['views'] for i in j['items']}, dtype='float64')
        time.sleep(.1)
    df = pd.DataFrame(colunas).sort_index()
    idx = pd.date_range(pd.Timestamp(inicio), fim, freq='MS' if por == 'mes' else 'D')
    df = df.reindex(idx).fillna(0).astype(int)
    df.index.name = 'mês' if por == 'mes' else 'dia'
    return df


def texto(titulo):
    """Texto puro de um verbete (sem tabelas nem imagens)."""
    t = titulo_oficial(titulo)
    j = _get('https://pt.wikipedia.org/w/api.php', dict(action='query', prop='extracts', explaintext=1, titles=t,
                                                         format='json', formatversion=2))
    return j['query']['pages'][0].get('extract', '')


def fmt(v, casas=0):
    """Número no formato brasileiro: 1.234,5"""
    return f'{v:,.{casas}f}'.replace(',', '_').replace('.', ',').replace('_', '.')


def baixar_csv(df, nome):
    df.to_csv(nome)
    try:
        from google.colab import files
        files.download(nome)
    except ImportError:
        print('Arquivo salvo:', nome)


print('Pronto. Funções disponíveis: visitas(), texto(), titulo_oficial(), fmt(), baixar_csv().')

## Passo 1. Os dados

In [ ]:
verbete = 'Fração'  # @param {type:"string"}
df = visitas(verbete, inicio='2016-01-01')
serie = df.iloc[:, 0]
serie = serie[serie.gt(0).idxmax():]  # começa no primeiro mês com visitas
ax = serie.plot(linewidth=2)
ax.set_title(f'{serie.name}: visitas por mês'); ax.set_ylabel('Visitas')
plt.show()

## Passo 2. Treino e teste
Guardamos os últimos meses como **teste**: o modelo não pode olhar para eles enquanto aprende. É como uma prova com questões que o aluno não viu antes. Se testássemos nos mesmos dados do treino, o modelo poderia apenas decorar.

In [ ]:
meses_de_teste = 12  # @param {type:"slider", min:6, max:24, step:1}
treino, teste = serie[:-meses_de_teste], serie[-meses_de_teste:]
ax = treino.plot(label='treino', linewidth=2)
teste.plot(ax=ax, label='teste', linewidth=2)
ax.axvline(teste.index[0], color='#111', linestyle='--', linewidth=1)
ax.legend(); ax.set_title('Separação entre treino e teste'); plt.show()
print(f'Treino: {len(treino)} meses. Teste: {len(teste)} meses.')

## Passo 3. Três modelos
1. **Ingênuo:** o próximo mês será igual ao último mês conhecido.
2. **Sazonal:** cada mês será igual ao mesmo mês do ano anterior.
3. **Regressão:** uma reta de tendência mais um ajuste para cada mês do ano, aprendidos com o treino.

A medida de erro é o **erro absoluto médio** (EAM): em média, quantas visitas a previsão errou por mês.

In [ ]:
MESES = ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez']

def matriz(indice, t0):
    """Colunas: intercepto, tendência (meses desde o início) e um ajuste para cada mês de fev a dez."""
    t = np.array([(d.year - t0.year) * 12 + d.month - t0.month for d in indice])
    mes = np.array([d.month for d in indice])
    return np.column_stack([np.ones(len(t)), t] + [(mes == m).astype(float) for m in range(2, 13)])

def prever(modelo, treino, indice_teste):
    if modelo == 'ingênuo':
        return pd.Series(treino.iloc[-1], index=indice_teste)
    if modelo == 'sazonal':
        return pd.Series([treino.get(d - pd.DateOffset(years=1), treino.iloc[-1]) for d in indice_teste], index=indice_teste)
    X = matriz(treino.index, treino.index[0])
    coef, *_ = np.linalg.lstsq(X, treino.values, rcond=None)
    return pd.Series(matriz(indice_teste, treino.index[0]) @ coef, index=indice_teste).clip(lower=0)

previsoes = {m: prever(m, treino, teste.index) for m in ['ingênuo', 'sazonal', 'regressão']}
erros = pd.Series({m: (p - teste).abs().mean() for m, p in previsoes.items()}, name='erro absoluto médio')
ax = teste.plot(color='#111', linewidth=3, label='o que aconteceu')
for m, p in previsoes.items():
    p.plot(ax=ax, linewidth=2, linestyle='--', label=f'{m} (erro médio {fmt(erros[m])})')
ax.legend(); ax.set_title('Previsões contra o que aconteceu no período de teste'); plt.show()
erros.sort_values().to_frame()

## Passo 4. Explicabilidade: o que o modelo aprendeu?
A regressão tem 13 parâmetros, e cada um tem um significado:
- **intercepto:** o ponto de partida da reta;
- **tendência:** quantas visitas a mais (ou a menos) por mês, em média, ao longo dos anos;
- **ajuste de cada mês:** quanto aquele mês fica acima ou abaixo de janeiro, descontada a tendência.

Um modelo com poucos parâmetros que dá para explicar um a um é chamado de **interpretável**. Modelos grandes, como as redes neurais, têm milhões de parâmetros e precisam de outras técnicas para serem explicados.

In [ ]:
X = matriz(treino.index, treino.index[0])
coef, *_ = np.linalg.lstsq(X, treino.values, rcond=None)
ajustes = pd.Series([0] + list(coef[2:]), index=MESES)
print(f'Intercepto: {fmt(coef[0])} visitas')
print(f'Tendência: {"+" if coef[1] >= 0 else ""}{fmt(coef[1], 1)} visitas por mês')
ax = ajustes.plot(kind='bar', color=[CORES[0] if v >= 0 else CORES[1] for v in ajustes], edgecolor='#111')
ax.set_title('Ajuste de cada mês em relação a janeiro'); ax.set_ylabel('Visitas'); plt.xticks(rotation=0); plt.show()

**Para discutir:** os meses com ajuste positivo coincidem com o ano letivo? Para um verbete de lazer, o padrão seria o mesmo?

## Passo 5. Sementes (seed) e reprodutibilidade
Para saber o quanto confiar na previsão, sorteamos erros do passado e somamos à previsão muitas vezes (isso se chama *bootstrap*). Como há sorteio, o resultado muda a cada execução, a não ser que fixemos a **semente** do gerador de números aleatórios. Mude a semente e o número de sorteios e veja o que acontece com o intervalo.

In [ ]:
residuos = (treino - pd.Series(X @ coef, index=treino.index)).values

@interact(semente=widgets.IntSlider(value=42, min=0, max=100), sorteios=widgets.IntSlider(value=200, min=20, max=2000, step=20))
def intervalo(semente, sorteios):
    rng = np.random.default_rng(semente)
    base = previsoes['regressão'].values
    sims = base + rng.choice(residuos, size=(sorteios, len(base)))
    baixo, alto = np.percentile(sims, [5, 95], axis=0)
    fig, ax = plt.subplots()
    ax.plot(teste.index, teste.values, color='#111', linewidth=3, label='o que aconteceu')
    ax.plot(teste.index, base, color=CORES[0], linewidth=2, linestyle='--', label='regressão')
    ax.fill_between(teste.index, baixo.clip(0), alto, color=CORES[4], alpha=.3, label='90% dos sorteios')
    dentro = ((teste.values >= baixo) & (teste.values <= alto)).mean()
    ax.set_title(f'Semente {semente}, {sorteios} sorteios: {dentro:.0%} dos meses dentro do intervalo')
    ax.legend(); plt.show()

Com a mesma semente, o resultado se repete exatamente: é isso que permite que outra pessoa confira um experimento. Com mais sorteios, o intervalo muda menos de uma semente para outra.

## Passo 6. A diferença entre dois modelos é significativa?
Um modelo errou menos que o outro no teste. Mas com só 12 meses, a vantagem pode ser sorte. O **teste de permutação** responde: se os dois modelos fossem igualmente bons, com que frequência veríamos uma diferença tão grande só por acaso? Essa frequência é o **valor-p**. Um valor-p pequeno (por exemplo, menor que 0,05) indica que a diferença provavelmente não é acaso.

In [ ]:
modelo_a = 'regressão'  # @param ["ingênuo", "sazonal", "regressão"]
modelo_b = 'ingênuo'  # @param ["ingênuo", "sazonal", "regressão"]
semente = 7  # @param {type:"integer"}

ea = (previsoes[modelo_a] - teste).abs().values
eb = (previsoes[modelo_b] - teste).abs().values
diff = eb - ea
observada = diff.mean()
rng = np.random.default_rng(semente)
sinais = rng.choice([-1, 1], size=(10000, len(diff)))
simuladas = (sinais * diff).mean(axis=1)
valor_p = (np.abs(simuladas) >= abs(observada)).mean()

plt.hist(simuladas, bins=50, color=CORES[4], edgecolor='#111')
plt.axvline(observada, color=CORES[1], linewidth=2.5, label='diferença observada')
plt.axvline(-observada, color=CORES[1], linewidth=1, linestyle='--')
plt.title(f'{modelo_a} contra {modelo_b}: valor-p = {fmt(valor_p, 3)}'); plt.legend(); plt.show()
print(f'Em média, {modelo_a} errou {fmt(observada)} visitas a menos por mês que {modelo_b}.')
print('Diferença significativa (valor-p < 0,05).' if valor_p < .05 else 'Não dá para afirmar que a diferença não é acaso (valor-p >= 0,05).')

## Sua vez
1. Troque o verbete por um de lazer (por exemplo, *Minecraft*) e compare os ajustes por mês com os de *Fração*.
2. Qual modelo ganha com 6 meses de teste? E com 24?
3. Rode o teste de permutação com outras sementes. O valor-p muda muito? Por quê?

In [ ]:
# escreva aqui

## Para fechar
- Um modelo que acerta o passado não garante acertar o futuro: por isso existe o teste.
- Explicar um modelo é dizer o que cada parâmetro faz. Quanto mais parâmetros, mais difícil.
- Resultados com sorteio só são confiáveis se forem reproduzíveis (semente fixa) e se a diferença passar por um teste de significância.